# Selección de decaimientos de muón (doble pulso en un centellador)

Lee todos los `.txt` de `muones/data` (incluidas subcarpetas), y selecciona los eventos en los que **un mismo canal** (u1, u2 o u3) tiene **dos pulsos** por encima de un umbral. El umbral de cada evento se calcula a partir de lo que alcanza el pulso de u1.

Hay dos modos (variable `MODO` en la celda de parámetros):
- `"suavizado"`: promedio móvil de 6 muestras (10 ns, anula el ruido de ~100 MHz) y resta de la línea de base (mediana). Más sensible a pulsos chicos.
- `"simple"`: voltaje crudo, umbral directo en mV. Anda bien para pulsos grandes (> ~70 mV).

Este notebook tiene que estar en `muones/analisis`; lee de `muones/data` y guarda en `muones/salida_decaimientos`.

In [ ]:
import glob
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

CANALES = ["u1", "u2", "u3"]

# Carpeta muones: el notebook está en muones/analisis, así que se sube un nivel
cwd = Path.cwd().resolve()
CARPETA_MUONES = cwd.parent if cwd.name == "analisis" else cwd
CARPETA_DATA = CARPETA_MUONES / "data"
CARPETA_SALIDA = CARPETA_MUONES / "salida_decaimientos"

print("Directorio de trabajo:", cwd)
print("Leo datos de        :", CARPETA_DATA, "(existe)" if CARPETA_DATA.is_dir() else "(NO EXISTE)")
print("Guardo resultados en:", CARPETA_SALIDA)

## Parámetros

In [ ]:
MODO = "suavizado"          # "suavizado" o "simple"

REF = "u1"                  # canal cuyo pico fija el umbral de cada evento
T_MUERTO = 40.0             # ns: separación mínima entre dos pulsos del mismo canal
N_SUAVIZADO = 6             # muestras del promedio móvil (6 = 10 ns). Solo modo "suavizado"

# umbral = max(FRAC * profundidad del pulso de REF, UMBRAL_MIN)   [mV]
if MODO == "suavizado":
    FRAC, UMBRAL_MIN = 0.25, 30.0
else:
    FRAC, UMBRAL_MIN = 0.5, 70.0   # el ruido crudo llega a ~ -62 mV

GUARDAR_PLOTS = False       # True: guarda un PNG por evento seleccionado
MAX_PLOTS_EN_PANTALLA = 10  # cuántos eventos seleccionados mostrar acá abajo

## Funciones

In [ ]:
def leer_eventos(path):
    """Lee un archivo con varios eventos. Devuelve lista de dicts con header, num, ts y data (N,4)."""
    eventos, actual = [], None
    with open(path, "r", errors="replace") as f:
        for linea in f:
            s = linea.strip()
            if not s or s.startswith("#") or s.startswith("t[ns]"):
                continue
            if s.startswith("Event"):
                partes = s.split()
                try:
                    num = int(partes[1].lstrip("#"))
                except (IndexError, ValueError):
                    num = -1
                ts = ""
                for k, p in enumerate(partes):
                    if p.startswith("ts="):
                        ts = " ".join([p[3:]] + ([partes[k + 1]] if k + 1 < len(partes) and ":" in partes[k + 1] else []))
                actual = {"header": s, "num": num, "ts": ts, "filas": []}
                eventos.append(actual)
            elif actual is not None:
                try:
                    actual["filas"].append([float(x) for x in s.split()])
                except ValueError:
                    pass
    for e in eventos:
        e["data"] = np.array(e.pop("filas"), dtype=float)
    return [e for e in eventos if e["data"].ndim == 2 and e["data"].shape[0] > 10]


def procesar_evento(data):
    """Devuelve t, s, a_ref, umbral, picos.
    s[i] es la señal del canal i como profundidad positiva (pulso hacia arriba):
      - modo suavizado: (mediana - promedio móvil)
      - modo simple:    -voltaje crudo
    """
    t = data[:, 0]
    dt = np.median(np.diff(t))
    s = np.empty((len(CANALES), len(t)))
    if MODO == "suavizado":
        kernel = np.ones(N_SUAVIZADO) / N_SUAVIZADO
        for i in range(len(CANALES)):
            v = np.convolve(data[:, 1 + i], kernel, mode="same")
            s[i] = np.median(v) - v
        borde = N_SUAVIZADO
    else:
        for i in range(len(CANALES)):
            s[i] = -data[:, 1 + i]
        borde = 0

    sl = slice(borde, len(t) - borde)
    a_ref = s[CANALES.index(REF), sl].max()
    umbral = max(FRAC * a_ref, UMBRAL_MIN)

    distancia = max(1, int(round(T_MUERTO / dt)))
    picos = {}
    for i, c in enumerate(CANALES):
        idx, _ = find_peaks(s[i], height=umbral, distance=distancia)
        picos[c] = idx[(idx >= borde) & (idx < len(t) - borde)]
    return t, s, a_ref, umbral, picos


def escribir_evento(f, e):
    f.write(e["header"] + "\n")
    f.write("  t[ns]  u1[mV]  u2[mV]  u3[mV]\n")
    for fila in e["data"]:
        f.write(f"{fila[0]:8.3f} {fila[1]:7.1f} {fila[2]:7.1f} {fila[3]:7.1f}\n")


def graficar(e, t, s, umbral, picos, canales_ok=(), path_png=None):
    """Grafica el evento (crudo en gris y señal analizada en azul) con el umbral y los picos."""
    fig, ax = plt.subplots(len(CANALES), 1, figsize=(11, 7), sharex=True)
    for i, c in enumerate(CANALES):
        ax[i].plot(t, e["data"][:, 1 + i], color="0.7", lw=0.7, label=f"{c} crudo")
        if MODO == "suavizado":
            base = np.median(np.convolve(e["data"][:, 1 + i], np.ones(N_SUAVIZADO) / N_SUAVIZADO, mode="same"))
            y = base - s[i]
        else:
            base, y = 0.0, -s[i]
        ax[i].plot(t, y, "C0", lw=1.2, label=f"{c} analizado")
        ax[i].axhline(base - umbral, color="r", ls="--", lw=0.8, label=f"umbral ({umbral:.0f} mV)")
        if len(picos[c]):
            ax[i].plot(t[picos[c]], y[picos[c]], "rv")
        ax[i].set_ylabel(f"{c} [mV]")
        ax[i].legend(loc="lower right", fontsize=7)
        if c in canales_ok:
            ax[i].set_title(f"{c}: {len(picos[c])} pulsos", fontsize=9, loc="left")
    ax[-1].set_xlabel("t [ns]")
    fig.suptitle(e["header"], fontsize=9)
    fig.tight_layout()
    if path_png is not None:
        fig.savefig(path_png, dpi=90)
    return fig

## Correr el análisis sobre todos los archivos de `data`

In [ ]:
archivos = sorted(glob.glob(str(CARPETA_DATA / "**" / "*.txt"), recursive=True))
if not archivos:
    raise FileNotFoundError(f"No encontré archivos .txt en {CARPETA_DATA}. ¿Descomprimiste los datos ahí?")

CARPETA_SALIDA.mkdir(exist_ok=True)
(CARPETA_SALIDA / "por_archivo").mkdir(exist_ok=True)
if GUARDAR_PLOTS:
    (CARPETA_SALIDA / "plots").mkdir(exist_ok=True)

seleccionados = []   # (nombre_base, evento, t, s, umbral, picos, canales_ok)
filas_resumen = []
n_total = 0

for path in archivos:
    base = os.path.splitext(os.path.basename(path))[0]
    eventos = leer_eventos(path)
    sel_archivo = []
    for e in eventos:
        n_total += 1
        t, s, a_ref, umbral, picos = procesar_evento(e["data"])
        ok = [c for c in CANALES if len(picos[c]) >= 2]
        if not ok:
            continue
        sel_archivo.append(e)
        seleccionados.append((base, e, t, s, umbral, picos, ok))
        for c in ok:
            i, p = CANALES.index(c), picos[c]
            filas_resumen.append(dict(archivo=base, evento=e["num"], ts=e["ts"], canal=c,
                                      umbral_mV=umbral, A_ref_mV=a_ref,
                                      t1_ns=t[p[0]], A1_mV=s[i][p[0]],
                                      t2_ns=t[p[1]], A2_mV=s[i][p[1]],
                                      dt_ns=t[p[1]] - t[p[0]], n_pulsos=len(p)))
    if sel_archivo:   # un archivo por cada archivo de entrada que tiene decaimientos
        with open(CARPETA_SALIDA / "por_archivo" / f"{base}_decaimientos.txt", "w") as fa:
            fa.write(f"# Eventos con doble pulso de {os.path.basename(path)}\n# Columns: t[ns] u1[mV] u2[mV] u3[mV]\n")
            for e in sel_archivo:
                escribir_evento(fa, e)
    print(f"{os.path.basename(path)}: {len(eventos)} eventos, {len(sel_archivo)} seleccionados")

# archivo con todos los eventos seleccionados + resumen en csv
with open(CARPETA_SALIDA / "eventos_decaimiento.txt", "w") as f:
    f.write("# Eventos con doble pulso en un mismo canal (candidatos a decaimiento de muon)\n")
    f.write("# Columns: t[ns] u1[mV] u2[mV] u3[mV]\n")
    for _, e, *_ in seleccionados:
        escribir_evento(f, e)

campos = ["archivo", "evento", "ts", "canal", "umbral_mV", "A_ref_mV", "t1_ns", "A1_mV", "t2_ns", "A2_mV", "dt_ns", "n_pulsos"]
with open(CARPETA_SALIDA / "resumen.csv", "w") as f:
    f.write(",".join(campos) + "\n")
    for r in filas_resumen:
        f.write(",".join(f"{r[k]:.1f}" if isinstance(r[k], (float, np.floating)) else str(r[k]) for k in campos) + "\n")

print(f"\nModo: {MODO} | umbral = max({FRAC} x |{REF}|, {UMBRAL_MIN} mV)")
print(f"Total: {len(seleccionados)} de {n_total} eventos seleccionados")
print("Resultados en:", CARPETA_SALIDA)

## Tabla de candidatos

In [ ]:
if not filas_resumen:
    print("Ningún evento seleccionado con estos parámetros.")
else:
    print(f"{'archivo':30s} {'ev':>5s} {'canal':>5s} {'t1[ns]':>8s} {'A1[mV]':>8s} {'t2[ns]':>8s} {'A2[mV]':>8s} {'dt[ns]':>8s}")
    for r in filas_resumen:
        print(f"{r['archivo'][:30]:30s} {r['evento']:5d} {r['canal']:>5s} {r['t1_ns']:8.1f} {r['A1_mV']:8.1f} "
              f"{r['t2_ns']:8.1f} {r['A2_mV']:8.1f} {r['dt_ns']:8.1f}")

## Gráficos de los eventos seleccionados

Conviene mirarlos para ajustar `FRAC` y `UMBRAL_MIN` en la celda de parámetros.

In [ ]:
for k, (base, e, t, s, umbral, picos, ok) in enumerate(seleccionados):
    path_png = None
    if GUARDAR_PLOTS:
        path_png = CARPETA_SALIDA / "plots" / f"{base}_ev{e['num']}.png"
    fig = graficar(e, t, s, umbral, picos, ok, path_png)
    if k < MAX_PLOTS_EN_PANTALLA:
        plt.show()
    else:
        plt.close(fig)
if len(seleccionados) > MAX_PLOTS_EN_PANTALLA:
    print(f"(mostré {MAX_PLOTS_EN_PANTALLA} de {len(seleccionados)}; el resto no se muestra acá)")

## Revisar un evento puntual (opcional)

Útil para ver por qué un evento entró o no. Cambiá el nombre del archivo y el número de evento.

In [ ]:
ARCHIVO_REVISAR = None   # por ejemplo "run_20261001_141452_idx0_selected"
EVENTO_REVISAR = 2       # número que aparece en "Event #N"

if ARCHIVO_REVISAR is not None:
    ruta = [a for a in archivos if os.path.splitext(os.path.basename(a))[0] == ARCHIVO_REVISAR][0]
    e = [ev for ev in leer_eventos(ruta) if ev["num"] == EVENTO_REVISAR][0]
    t, s, a_ref, umbral, picos = procesar_evento(e["data"])
    print(f"A_ref = {a_ref:.1f} mV, umbral = {umbral:.1f} mV, pulsos por canal:", {c: len(picos[c]) for c in CANALES})
    graficar(e, t, s, umbral, picos, [c for c in CANALES if len(picos[c]) >= 2])
    plt.show()